In [38]:
import os
import sys
import yaml
import pandas as pd
import numpy as np

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import writing_tools as wt

from matplotlib import pyplot as plt
from utils import parse_casenum

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

os.environ['LOKY_MAX_CPU_COUNT'] = '1' # because of windows core count warning

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)

N_CLUSTERS = 3
N_COMPONENTS = 10

In [39]:
df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "analysis_data_w_embeddings.parquet"))

In [40]:
flat_df = {}
df['canonical_casenum'] = df['title'].apply(lambda x: parse_casenum(x)['canonical_casenum'])
for idx, row in df.iterrows():
    date = pd.to_datetime(row['date'])
    project_result = row["project_result"]
    canonical_casenum = row['canonical_casenum']
    casenum = row['title']
    if canonical_casenum not in flat_df:
        flat_df[canonical_casenum] = {
            "hearing_dates": [],
            "project_results": [],
            "casenums": []
        }
    flat_df[canonical_casenum]["hearing_dates"].append(date)
    flat_df[canonical_casenum]["project_results"].append(project_result)
    flat_df[canonical_casenum]["casenums"].append(casenum)

flat_df = pd.DataFrame(flat_df, index=['hearing_dates', 'project_results', 'casenums']).T.reset_index().rename(columns={'index': 'canonical_casenum'})

In [41]:
flat_df['n_hearings'] = flat_df['hearing_dates'].apply(len)
flat_df['last_result'] = flat_df['project_results'].apply(lambda x: x[-1])
flat_df['first_result'] = flat_df['project_results'].apply(lambda x: x[0])
flat_df["average_hearing_interval"] = flat_df['hearing_dates'].apply(lambda x: np.mean(np.diff(sorted(x)).astype('timedelta64[D]')) if len(x) > 1 else np.nan)
mask = (flat_df["n_hearings"]>1) & (flat_df["first_result"]=="DELAYED") 
flat_df.loc[mask, "average_hearing_interval"].mean()

Timedelta('56 days 02:10:54.545454545')

In [42]:
NumDelayedCases = (flat_df["first_result"]=="DELAYED").sum()

mask = (flat_df["n_hearings"]>1) & (flat_df["first_result"]=="DELAYED")
PctHeardAgain = mask.sum() / NumDelayedCases
TimeBetweenHearings = flat_df.loc[mask, "average_hearing_interval"].mean() / pd.to_timedelta(1, unit='D')

PctUltimatelyApproved = ((flat_df["last_result"]=="APPROVED") & (flat_df["first_result"]=="DELAYED")).sum() / NumDelayedCases

RESULTS = {
    "NumDelayedCases": f"{NumDelayedCases:.0f}",
    "PctHeardAgain": f"{PctHeardAgain*100:.1f}",
    "TimeBetweenHearings": f"{TimeBetweenHearings:.0f}",
    "PctUltimatelyApproved": f"{PctUltimatelyApproved*100:.1f}"
}

RESULTS


{'NumDelayedCases': '71',
 'PctHeardAgain': '93.0',
 'TimeBetweenHearings': '56',
 'PctUltimatelyApproved': '29.6'}